In [ ]:
import numpy as np, pandas as pd, os, pickle, json
from pathlib import Path
from sklearn.metrics import mean_absolute_error

curr_path_file = Path("current_run.txt")
run_rel = curr_path_file.read_text().strip()
RUN_PATH = Path("experiments") / Path(run_rel)
RUN_PATH.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = RUN_PATH / "config.json"
CONFIG = json.load(open(CONFIG_PATH))
CACHE_DIR = RUN_PATH / "cache"

MODEL_NAME = "Naive-12Mean"
PRED_DIR = RUN_PATH / "predictions" / MODEL_NAME
RESULTS_DIR = RUN_PATH / "results" / MODEL_NAME
MODELS_DIR = RUN_PATH / "models" / MODEL_NAME
for p in [PRED_DIR, RESULTS_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

all_tags = CONFIG["tags"]
horizons = CONFIG["horizons"]
region_targets = CONFIG["regions"]
window = CONFIG["window"]

run_tags = ['P','PC','PCS','PCSB','PCSBI','PCSBIG']
tags = [t for t in run_tags if t in all_tags]

df_w = pd.read_csv(CONFIG["data_paths"]["price_series"], index_col="date", parse_dates=True)

print(f"Running Naive-12Mean on {len(tags)} tags × {len(region_targets)} regions × {len(horizons)} horizons")
print(f"Experiment: {CONFIG.get('exp_name','')} / {CONFIG.get('run_stamp','')}")

scores_mean_tag  = {t:{r:[np.inf]*len(horizons) for r in region_targets} for t in tags}
scores_std_tag   = {t:{r:[0.0]*len(horizons) for r in region_targets} for t in tags}

total = len(tags)*len(region_targets)*len(horizons)
done = 0
tick_every = 200

for tag in tags:
    flat_npz_path = CACHE_DIR / "features_flat" / f"{tag}.npz"
    folds_pkl_path = CACHE_DIR / "folds" / f"{tag}.pkl"

    if not flat_npz_path.exists() or not folds_pkl_path.exists():
        print(f"[Skip] Missing cache for tag {tag}")
        continue

    flat_npz = np.load(flat_npz_path, allow_pickle=False)
    folds_map = pickle.load(open(folds_pkl_path,"rb"))

    for reg in region_targets:
        if reg not in df_w.columns:
            done += len(horizons)
            if done % tick_every == 0 or done == total:
                print(f"Progress {done}/{total}")
            continue

        all_rows = []
        rolling_mean = df_w[reg].rolling(12).mean().shift(1)

        for hi, h in enumerate(horizons):
            kb = f"{reg}__h{h}"
            y_key = f"{kb}__y"
            idx_key = f"{kb}__idx"
            if y_key not in flat_npz or idx_key not in flat_npz:
                done += 1
                if done % tick_every == 0 or done == total:
                    print(f"Progress {done}/{total}")
                continue

            yf = flat_npz[y_key]
            idx = pd.to_datetime(flat_npz[idx_key])
            folds = folds_map.get(kb, [])
            maes = []

            for fold_i, f in enumerate(folds):
                va = f[1]
                naive_pred = rolling_mean.loc[idx[va]].values
                mae = mean_absolute_error(yf[va], naive_pred)
                maes.append(mae)
                all_rows.append(pd.DataFrame({
                    "date": idx[va],
                    "model": MODEL_NAME,
                    "horizon": h,
                    "fold": fold_i,
                    "actual": yf[va],
                    "predicted": naive_pred
                }))

            if maes:
                scores_mean_tag[tag][reg][hi] = float(np.mean(maes))
                scores_std_tag[tag][reg][hi] = float(np.std(maes))

            done += 1
            if done % tick_every == 0 or done == total:
                print(f"Progress {done}/{total}")

        if all_rows:
            out_path = PRED_DIR / f"{tag}__{reg}__cv.parquet"
            pd.concat(all_rows, ignore_index=True).to_parquet(out_path, index=False)

rows = []
for tag in tags:
    for reg in region_targets:
        for hi, h in enumerate(horizons):
            rows.append({
                "tag": tag,
                "region": reg,
                "model": MODEL_NAME,
                "horizon": h,
                "mae_mean": scores_mean_tag[tag][reg][hi],
                "mae_std": scores_std_tag[tag][reg][hi]
            })
pd.DataFrame(rows).to_csv(RESULTS_DIR / "scores_tag_all_cv.csv", index=False)

best = []
for reg in region_targets:
    for hi, h in enumerate(horizons):
        vals = [scores_mean_tag[t][reg][hi] for t in tags if np.isfinite(scores_mean_tag[t][reg][hi])]
        best.append({
            "region": reg,
            "model": MODEL_NAME,
            "horizon": h,
            "best_mae_mean": float(np.min(vals)) if len(vals) > 0 else np.inf
        })
pd.DataFrame(best).to_csv(RESULTS_DIR / "scores_best_cv.csv", index=False)

with open(RESULTS_DIR / "scores_pickle_cv.pkl","wb") as f:
    pickle.dump({"scores_mean_tag": scores_mean_tag, "scores_std_tag": scores_std_tag}, f)

print(f"{MODEL_NAME} results saved under: {RUN_PATH}")


Running Naive-12Mean on 6 tags × 19 regions × 9 horizons
Experiment: additive_tags_main_global_aug / 2026_01_26_123128
Progress 200/1026
Progress 400/1026
Progress 600/1026
Progress 800/1026
Progress 1000/1026
Progress 1026/1026
Naive-12Mean results saved under: experiments\additive_tags_main_global_aug\2026_01_26_123128
